# recursion_limit과 명시적 종료

그래프의 반복이 끝나지 않을 때 실행 한도와 상태 기반 종료 조건은 각각 어떤 역할을 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. API 키 값은 출력하지 않습니다.

## 패키지 준비

고정한 LangGraph 버전을 확인하고 그래프 구성 요소를 불러옵니다.

In [1]:
# 패키지 준비에 필요한 연산을 실행합니다.
%pip install -q langgraph==1.2.11
from importlib.metadata import version
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
langgraph 1.2.11


## Before: 종료 없는 순환 그래프

count를 늘린 뒤 다시 같은 노드로 이동하는 종료 없는 그래프를 만듭니다.

In [2]:
# Before: 종료 없는 순환 그래프에 필요한 연산을 실행합니다.
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START

class LoopState(TypedDict):
    count: int
    trace: list[str]

def tick(state: LoopState) -> dict:
    next_count = state['count'] + 1
    return {'count': next_count, 'trace': state['trace'] + [f'tick:{next_count}']}

before_builder = StateGraph(LoopState)
before_builder.add_node('tick', tick)
before_builder.add_edge(START, 'tick')
before_builder.add_edge('tick', 'tick')
before_graph = before_builder.compile()

## Before: 실제 한도 초과 확인

recursion_limit=4로 실행해 GraphRecursionError가 발생하는지 확인합니다.

In [3]:
# Before: 실제 한도 초과 확인에 필요한 연산을 실행합니다.
from langgraph.errors import GraphRecursionError

try:
    before_graph.invoke({'count': 0, 'trace': []}, {'recursion_limit': 4})
except GraphRecursionError as error:
    error_name = type(error).__name__
    print(error_name)
    print(str(error).splitlines()[0])
assert error_name == 'GraphRecursionError'

GraphRecursionError
Recursion limit of 4 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.


## After: 상태 기반 종료 조건 추가

count가 3에 도달하면 END를 반환하는 stop_or_continue 조건을 추가합니다.

In [4]:
# After: 상태 기반 종료 조건 추가에 필요한 연산을 실행합니다.
from typing import Literal
from langgraph.graph import END

def stop_or_continue(state: LoopState) -> Literal['tick', '__end__']:
    return END if state['count'] >= 3 else 'tick'

after_builder = StateGraph(LoopState)
after_builder.add_node('tick', tick)
after_builder.add_edge(START, 'tick')
after_builder.add_conditional_edges('tick', stop_or_continue)
after_graph = after_builder.compile()

## After: 정상 종료와 실행 흔적 확인

같은 실행 한도에서 count 값 3과 세 번의 tick 기록을 반환하는지 확인합니다.

In [5]:
# After: 정상 종료와 실행 흔적 확인에 필요한 연산을 실행합니다.
after = after_graph.invoke({'count': 0, 'trace': []}, {'recursion_limit': 4})
print('count:', after['count'])
print('trace:', after['trace'])
print(after_graph.get_graph().draw_mermaid())
assert after['count'] == 3
assert len(after['trace']) == 3

count: 3
trace: ['tick:1', 'tick:2', 'tick:3']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	tick(tick)
	__end__([<p>__end__</p>]):::last
	__start__ --> tick;
	tick -.-> __end__;
	tick -.-> tick;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

